### Using this notebook for hyperparameter tuning

In [10]:
#importing libraries

import numpy as np
import pandas as pd
import tensorflow as tf
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from sklearn.model_selection import GridSearchCV



In [2]:
#importing dataset into dataframe

data = pd.read_csv("churn_modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#data preprocessing
data.drop(columns=['RowNumber', 'Surname', 'CustomerId'], inplace=True)

#splitting data into x and y
x = data.drop(columns=['Exited'])
y = data['Exited']

#splitting data in train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

#encoding categorical features
label_encoder = LabelEncoder()
x_train['Gender'] = label_encoder.fit_transform(x_train['Gender'])
x_test['Gender'] = label_encoder.transform(x_test['Gender'])

onehot_encoder = OneHotEncoder()
geo_train_arr = onehot_encoder.fit_transform(x_train[['Geography']]).toarray()
geo_test_arr = onehot_encoder.transform(x_test[['Geography']]).toarray()

geo_train_df = pd.DataFrame(geo_train_arr, columns=onehot_encoder.get_feature_names_out())
geo_test_df = pd.DataFrame(geo_test_arr, columns=onehot_encoder.get_feature_names_out())

x_train = pd.concat([x_train.reset_index(drop=True), geo_train_df], axis=1)
x_test = pd.concat([x_test.reset_index(drop=True), geo_test_df], axis=1)

x_train.drop(columns=['Geography'], inplace=True)
x_test.drop(columns=['Geography'], inplace=True)

#scaling data features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [6]:
#saving ojects to pickle files

with open("onehot_encoder_hp.pkl", "wb") as file_obj:
    pickle.dump(onehot_encoder,file_obj)

with open("scaler_hp.pkl", "wb") as file_obj:
    pickle.dump(scaler, file_obj)

with open("label_encoder_hp.pkl", "wb") as file_obj:
    pickle.dump(label_encoder, file_obj)
        

In [8]:
#custom function to create model

def create_model(neurons:int, layers:int):
    "this function creates model based on neurons and layers passed"

    model = Sequential([
        Dense(units=neurons, activation="relu", input_shape=(x_train.shape[1], ))
        
    ])

    for _ in range(layers-1):
        model.add(Dense(units=neurons, activation="relu"))

    model.add(Dense(units=1, activation="sigmoid"))

    return model    

In [12]:
model = KerasClassifier(build_fn=create_model,neurons=32, loss="binary_crossentropy", layers=1, epochs=50, batch_size=10, verbose=True)

params_grid = {
    "neurons":[16, 32, 64, 128],
    "epochs": [50, 100],
    "layers": [1, 2]
}


grid_search = GridSearchCV(estimator=model, cv=3, param_grid=params_grid, verbose=1)
grid_result = grid_search.fit(x_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50


d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 863us/step - loss: 0.5135
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 811us/step - loss: 0.4226
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 801us/step - loss: 0.4031
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 864us/step - loss: 0.3871
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 895us/step - loss: 0.3738
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 814us/step - loss: 0.3632
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 865us/step - loss: 0.3558
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 807us/step - loss: 0.3503
Epoch 9/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 851us/step - loss: 0.3465
Epoch 10/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 839us/step - loss: 0.3434
Epoch 11/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 863us/step - loss: 0.3413
Epoch 12/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 885us/step - loss: 0.3394
Epoch 13/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 822us/step - loss: 0.3386
Epoch 14/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 881us/step - loss: 0.3375
Epoch 15/50
500/500 ━━━━━━━━━━━━━━━━━━━━

d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 815us/step - loss: 0.5155
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 840us/step - loss: 0.4463
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 818us/step - loss: 0.4291
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 817us/step - loss: 0.4146
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 828us/step - loss: 0.4000
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 843us/step - loss: 0.3878
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 833us/step - loss: 0.3781
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 839us/step - loss: 0.3711
Epoch 9/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 818us/step - loss: 0.3662
Epoch 10/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 868us/step - loss: 0.3621
Epoch 11/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 848us/step - loss: 0.3588
Epoch 12/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 813us/step - loss: 0.3568
Epoch 13/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 844us/step - loss: 0.3546
Epoch 14/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 867us/step - loss: 0.3530
Epoch 15/50
500/500 ━━━━━━━━━━━━━━━━━━━━

...

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4308
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3723
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3580
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3497
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3449
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3456
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3416
Epoch 8/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3365
Epoch 9/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3351
Epoch 10/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3336
Epoch 11/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3309
Epoch 12/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3295
Epoch 13/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3279
Epoch 14/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3245
Epoch 15/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step -

d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4140
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3551
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3417
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3346
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3301
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3270
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3250
Epoch 8/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3228
Epoch 9/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3191
Epoch 10/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3165
Epoch 11/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3150
Epoch 12/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3137
Epoch 13/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3110
Epoch 14/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3080
Epoch 15/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step -

d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.4275
Epoch 2/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 914us/step - loss: 0.3729
Epoch 3/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3557
Epoch 4/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3489
Epoch 5/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3449
Epoch 6/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3425
Epoch 7/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3420
Epoch 8/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3399
Epoch 9/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3391
Epoch 10/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3387
Epoch 11/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3377
Epoch 12/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3364
Epoch 13/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3353
Epoch 14/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3347
Epoch 15/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.332

In [13]:
grid_search.best_params_

{'epochs': 50, 'layers': 1, 'neurons': 128}

In [16]:
grid_search.best_score_

np.float64(0.8549333333333333)